# Vergelijking van de twee verschilscore-technieken

Zelfde data (heel Etten-Leur, 2022 → 2025), twee methoden naast elkaar:

| | **v1** | **v2** |
|---|---|---|
| Meetgebied | omsluitende box van het pand | alleen bínnen de pandcontour |
| Drempel | één stadsbrede drempel | per omgevingstype (stedelijk / buitengebied) |

Beide scores worden in één rekengang bepaald (zelfde verschilkaart, twee aggregaties),
dus dit kost niet meer tijd dan één gewone run. We vergelijken op: score-verschuiving,
recall op de eerlijke meetlat (nieuwbouw 2023–2024), samenstelling van de werkvoorraad,
en voorbeelden waar de technieken het oneens zijn.

Vereist: dezelfde downloads als het mutatierapport (`el_2022`, `el_2025`).

In [ ]:
import io, json, sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from shapely.geometry import shape

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'scripts'))
from common import laad_config
from pdok import fetch_bag_panden, make_session, wms_get_map

cfg = laad_config(REPO / 'config.yaml')
DATA = REPO / cfg['paden']['data']
RES = cfg['luchtfoto']['resolutie']
PX = cfg['luchtfoto']['tegelgrootte']
STAP = PX * RES

GEBIED = [101500, 395000, 107000, 400000]
OUD_LAAG, NIEUW_LAAG = '2022_orthoHR', '2025_orthoHR'
OUD_MAP, NIEUW_MAP = DATA / 'el_2022', DATA / 'el_2025'
BAG_ALLE = DATA / 'bag' / 'el_panden_alle.geojson'
DREMPEL_PCT = 99
STEDELIJK_VANAF = 6
MARKEER = '#FFD400'
BLAUW, GRIJS, AMBER = '#2563EB', '#94A3B8', '#F59E0B'
sessie = make_session()

for m in (OUD_MAP, NIEUW_MAP):
    assert (m / 'tiles.json').exists(), f'{m} ontbreekt — draai eerst de downloads (zie intro)'
idx_oud = json.loads((OUD_MAP / 'tiles.json').read_text())
idx_nieuw = json.loads((NIEUW_MAP / 'tiles.json').read_text())
if not BAG_ALLE.exists():
    feats = fetch_bag_panden(sessie, cfg['bag']['wfs_url'], tuple(GEBIED), alleen_in_gebruik=False)
    BAG_ALLE.write_text(json.dumps({'type': 'FeatureCollection', 'bbox_rd': GEBIED, 'features': feats}))
panden = json.loads(BAG_ALLE.read_text())['features']
nieuwbouw = [f for f in panden if 2022 <= (f['properties'].get('bouwjaar') or 0) <= 2025]
in_aanbouw = [f for f in panden if (f['properties'].get('bouwjaar') or 0) >= 2026
              or f['properties'].get('status') == 'Bouw gestart']
overig_bag = [f for f in panden if f['properties'].get('status') in
              ('Verbouwing pand', 'Sloopvergunning verleend', 'Bouwvergunning verleend')]
bag_verklaard_ids = {f['properties']['identificatie'] for f in nieuwbouw + in_aanbouw + overig_bag}
print(f'{len(panden)} BAG-panden, {len(set(idx_oud) & set(idx_nieuw))} tegelparen geladen')

## 1. Beide scores in één rekengang

Zelfde voorbewerking (blur, normalisatie, verschuivings-tolerant minimum); daarna per
pand twee aggregaties: gemiddelde over de **box** (v1) en over het **contourmasker** (v2).

In [ ]:
GRID_X, GRID_Y = GEBIED[0], GEBIED[1]
def tegel_id_voor(x, y):
    return f't_{int((x - GRID_X) // STAP):04d}_{int((y - GRID_Y) // STAP):04d}'

def laad_genorm(pad):
    beeld = Image.open(pad).convert('L').filter(ImageFilter.GaussianBlur(1.5))
    a = np.asarray(beeld, dtype=np.float32)
    return (a - a.mean()) / (a.std() + 1e-6)

per_tegel = defaultdict(list)
for f in panden:
    geom = shape(f['geometry'])
    if geom.area < 25:
        continue
    per_tegel[tegel_id_voor(geom.centroid.x, geom.centroid.y)].append((f, geom))

paren = [tid for tid in per_tegel if tid in idx_oud and tid in idx_nieuw]
VERSCHUIVINGEN = [(dx, dy) for dx in (-8, 0, 8) for dy in (-8, 0, 8)]
resultaten = []   # (pid, feature, geom, score_v1, score_v2, stratum)
from tqdm.auto import tqdm
for tid in tqdm(paren, desc='tegelparen scoren'):
    a = laad_genorm(OUD_MAP / idx_oud[tid]['image'])
    b = laad_genorm(NIEUW_MAP / idx_nieuw[tid]['image'])
    minverschil = np.full_like(a, np.inf)
    for dx, dy in VERSCHUIVINGEN:
        d = np.abs(np.roll(b, (dy, dx), axis=(0, 1)) - a)
        np.minimum(minverschil, d, out=minverschil)
    bbox = idx_oud[tid]['bbox']
    stratum = 'stedelijk' if idx_oud[tid].get('n_panden', 0) >= STEDELIJK_VANAF else 'buitengebied'
    for f, geom in per_tegel[tid]:
        gx0, gy0, gx1, gy1 = geom.bounds
        x0 = max(0, int((gx0 - bbox[0]) / RES)); x1 = min(PX, int((gx1 - bbox[0]) / RES))
        y0 = max(0, int((bbox[3] - gy1) / RES)); y1 = min(PX, int((bbox[3] - gy0) / RES))
        if x1 - x0 < 12 or y1 - y0 < 12:
            continue
        venster = minverschil[y0:y1, x0:x1]
        masker = Image.new('1', (x1 - x0, y1 - y0), 0)
        tekenaar = ImageDraw.Draw(masker)
        for poly in (geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]):
            punten = [((px - bbox[0]) / RES - x0, (bbox[3] - py) / RES - y0)
                      for px, py in poly.exterior.coords]
            tekenaar.polygon(punten, fill=1)
        binnen = np.asarray(masker, dtype=bool)
        s_v1 = float(venster.mean())
        s_v2 = float(venster[binnen].mean()) if binnen.sum() >= 100 else s_v1
        resultaten.append((f['properties']['identificatie'], f, geom, s_v1, s_v2, stratum))

v1 = np.array([r[3] for r in resultaten])
v2 = np.array([r[4] for r in resultaten])
strata = np.array([r[5] for r in resultaten])
print(f'{len(resultaten)} panden dubbel gescoord')

## 2. Hoe verschuiven de scores?

Elke stip is een pand. Onder de diagonaal = v2 (contour) lager dan v1 (box) — dat is
omgevingsruis die uit de meting verdwenen is. Verwachting: buitengebied zakt het sterkst.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 6))
for naam, kleur in (('stedelijk', BLAUW), ('buitengebied', AMBER)):
    m = strata == naam
    ax.scatter(v1[m], v2[m], s=4, alpha=0.25, color=kleur, label=naam, rasterized=True)
grens = max(v1.max(), v2.max())
ax.plot([0, grens], [0, grens], color='#555555', linewidth=1, linestyle='--')
ax.set_xlabel('score v1 (box)'); ax.set_ylabel('score v2 (contour)')
ax.set_title('Score per pand: v1 vs v2', loc='left', fontsize=11)
ax.legend(frameon=False, fontsize=9, markerscale=3)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(length=0)
plt.tight_layout(); plt.show()

for naam in ('stedelijk', 'buitengebied'):
    m = strata == naam
    print(f'{naam}: mediaan v1 {np.median(v1[m]):.3f} -> v2 {np.median(v2[m]):.3f} '
          f'(daling {(1 - np.median(v2[m]) / np.median(v1[m])):.0%})')

## 3. Detecties: wie vindt wat, en wat vinden ze samen?

In [ ]:
# v1: één globale drempel op de boxscores; v2: drempel per stratum op de contourscores.
drempel_v1 = float(np.percentile(v1, DREMPEL_PCT))
drempel_v2 = {naam: float(np.percentile(v2[strata == naam], DREMPEL_PCT))
              for naam in ('stedelijk', 'buitengebied')}

det_v1 = {r[0] for r in resultaten if r[3] >= drempel_v1}
det_v2 = {r[0] for r in resultaten if r[4] >= drempel_v2[r[5]]}
beide = det_v1 & det_v2
info = {r[0]: r for r in resultaten}

def buiten_aandeel(ids):
    return (sum(1 for i in ids if info[i][5] == 'buitengebied') / len(ids)) if ids else 0

print(f'v1: {len(det_v1)} detecties ({buiten_aandeel(det_v1):.0%} buitengebied)')
print(f'v2: {len(det_v2)} detecties ({buiten_aandeel(det_v2):.0%} buitengebied)')
print(f'eens over {len(beide)}; alleen-v1 {len(det_v1 - det_v2)}, alleen-v2 {len(det_v2 - det_v1)}')

# Recall op de eerlijke meetlat: nieuwbouw 2023-2024
meetlat = [f['properties']['identificatie'] for f in nieuwbouw
           if 2023 <= f['properties']['bouwjaar'] <= 2024]
meetlat = [i for i in meetlat if i in info]
recall_v1 = sum(1 for i in meetlat if i in det_v1) / max(1, len(meetlat))
recall_v2 = sum(1 for i in meetlat if i in det_v2) / max(1, len(meetlat))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.5, 3))
ax1.bar(['v1 (box,\nglobaal)', 'v2 (contour,\nper stratum)'],
        [recall_v1 * 100, recall_v2 * 100], color=[GRIJS, BLAUW], width=0.5)
for i, r in enumerate((recall_v1, recall_v2)):
    ax1.text(i, r * 100 + 1.5, f'{r:.0%}', ha='center', color='#555555', fontsize=10)
ax1.set_title(f'Recall op nieuwbouw 2023-2024 (n={len(meetlat)})', loc='left', fontsize=10)
ax1.set_ylabel('% gevangen')
ax2.bar(['v1', 'v2'], [buiten_aandeel(det_v1) * 100, buiten_aandeel(det_v2) * 100],
        color=[GRIJS, BLAUW], width=0.5)
for i, a in enumerate((buiten_aandeel(det_v1), buiten_aandeel(det_v2))):
    ax2.text(i, a * 100 + 1.5, f'{a:.0%}', ha='center', color='#555555', fontsize=10)
ax2.set_title('Aandeel buitengebied in de detecties', loc='left', fontsize=10)
ax2.set_ylabel('%')
for ax in (ax1, ax2):
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(length=0)
plt.tight_layout()
(DATA / 'mutaties_preview').mkdir(exist_ok=True)
fig.savefig(DATA / 'mutaties_preview' / 'vergelijking.jpg', dpi=110, bbox_inches='tight')
plt.show()

## 4. Waar zijn ze het oneens? (voor/na-voorbeelden)

- **Alleen v2**: veranderingen die v1 miste — vaak stedelijke panden die onder de
  (door buitengebied opgedreven) globale drempel bleven.
- **Alleen v1**: detecties die v2 laat vallen — vaak erf-/akkerruis in de box, of
  buitengebied-panden die tegen hun eigen (hogere) stratumdrempel niet meer opvallen.

In [ ]:
def beeldpaar_rond(geom):
    cx, cy = geom.centroid.x, geom.centroid.y
    tid = tegel_id_voor(cx, cy)
    if tid in idx_oud and tid in idx_nieuw:
        return (Image.open(OUD_MAP / idx_oud[tid]['image']).convert('RGB'),
                Image.open(NIEUW_MAP / idx_nieuw[tid]['image']).convert('RGB'),
                idx_oud[tid]['bbox'])
    halve = STAP / 2
    bbox = (cx - halve, cy - halve, cx + halve, cy + halve)
    oud, nieuw = (Image.open(io.BytesIO(wms_get_map(sessie, cfg['luchtfoto']['wms_url'],
                                                    laag, bbox, PX, PX))).convert('RGB')
                  for laag in (OUD_LAAG, NIEUW_LAAG))
    return oud, nieuw, list(bbox)

def markeer_en_crop(beeld, geom, bbox, marge_m=14):
    gx0, gy0, gx1, gy1 = geom.bounds
    x0, x1 = (gx0 - bbox[0]) / RES, (gx1 - bbox[0]) / RES
    y0, y1 = (bbox[3] - gy1) / RES, (bbox[3] - gy0) / RES
    kopie = beeld.copy()
    ImageDraw.Draw(kopie).rectangle([x0, y0, x1, y1], outline=MARKEER, width=4)
    m = marge_m / RES
    return kopie.crop((max(0, x0 - m), max(0, y0 - m), min(PX, x1 + m), min(PX, y1 + m)))

def toon_voor_na(rijen, titel):
    if not rijen:
        print('(geen voorbeelden)'); return
    fig, assen = plt.subplots(len(rijen), 2, figsize=(9, 4.4 * len(rijen)), squeeze=False)
    for (geom, onderschrift), (as_o, as_n) in zip(rijen, assen):
        oud, nieuw, bbox = beeldpaar_rond(geom)
        as_o.imshow(markeer_en_crop(oud, geom, bbox))
        as_n.imshow(markeer_en_crop(nieuw, geom, bbox))
        as_o.set_title(f'2022 — {onderschrift}', fontsize=9, loc='left')
        as_n.set_title('2025', fontsize=9, loc='left')
        as_o.axis('off'); as_n.axis('off')
    fig.suptitle(titel, fontsize=12)
    plt.tight_layout(); plt.show()

alleen_v2 = sorted((info[i] for i in det_v2 - det_v1), key=lambda r: -r[4])
alleen_v1 = sorted((info[i] for i in det_v1 - det_v2), key=lambda r: -r[3])

rijen = [(r[2], f'ALLEEN v2: contour {r[4]:.2f}, box {r[3]:.2f} ({r[5]})') for r in alleen_v2[:4]]
toon_voor_na(rijen, 'Gevonden door v2, gemist door v1')
rijen = [(r[2], f'ALLEEN v1: box {r[3]:.2f}, contour {r[4]:.2f} ({r[5]})') for r in alleen_v1[:4]]
toon_voor_na(rijen, 'Gevonden door v1, vervallen in v2 — is dit ruis of echt?')

## 5. Conclusie

In [ ]:
print(f'Vergelijking op {len(resultaten)} panden (drempel P{DREMPEL_PCT}):')
print(f'- Recall nieuwbouw 2023-2024: v1 {recall_v1:.0%} -> v2 {recall_v2:.0%}')
print(f'- Aandeel buitengebied in detecties: v1 {buiten_aandeel(det_v1):.0%} -> v2 {buiten_aandeel(det_v2):.0%}')
print(f'- Overlap: {len(beide)} gezamenlijk, {len(det_v1 - det_v2)} alleen-v1, {len(det_v2 - det_v1)} alleen-v2')
print()
print('Duiding: is de v2-recall hoger of gelijk terwijl het buitengebied-aandeel daalt,')
print('dan is v2 de betere zeef (zelfde vangst, minder ruis). Bekijk de alleen-v1-galerij')
print('om te bevestigen dat wat v2 laat vallen inderdaad omgevingsruis was.')

---
*Bevat gegevens van PDOK: Luchtfoto Beeldmateriaal Nederland (CC-BY 4.0) en de BAG.*